In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import xarray as xr
import os
import matplotlib as mpl
fs=10
mpl.rc('xtick', labelsize=fs)
mpl.rc('ytick', labelsize=fs)
mpl.rc('legend', fontsize=fs)
mpl.rc('axes', titlesize=fs)
mpl.rc('axes', labelsize=fs)
mpl.rc('figure', titlesize=fs)
mpl.rc('font', size=fs)
mpl.rc('font', family='sans-serif', weight='normal', style='normal')



In [2]:
def make_filename(path_run,folder, var='biol_T', res='d'):
    """Construct path prefix for local SHEM results given date object and paths dict"""
    prefix = os.path.join(path_run, f'{folder}/')
    fname = []
    for file in os.listdir(prefix):
        if (var in file) and ('_1'+res) in file:
            fname.append(file)
    if len(fname)>1:
        print('more than one file found') 
    
    return os.path.join(prefix, fname[0])

In [3]:
jjii = xr.open_dataset('~/MOAD/grid/grid_from_lat_lon_mask999.nc')
def finder(lati,loni):
    j = [jjii.jj.sel(lats=lati, lons=loni, method='nearest').item()][0]
    i = [jjii.ii.sel(lats=lati, lons=loni, method='nearest').item()][0]
    return j,i

In [4]:
cnode = [49.04005171084712, -123.42548556523623]
jj, ii = finder(cnode[0], cnode[1])

In [5]:
mask = xr.open_dataset('/home/jvalenti/MOAD/grid2/mesh_mask202108_TDV.nc')
coords=  xr.open_dataset('~/MOAD/grid/coordinates_seagrid_SalishSea201702.nc', decode_times=False)

In [6]:
runs = ['SHEM18','tuning/diat_pref','tuning/exc_hbac','tuning/exc_hbac_2','tuning/growth_flag','tuning/growth_flag_2','tuning/mort_hbac','tuning/pred_flag','tuning/remin','tuning/remin2','tuning/predmine','tuning/mort_hbac_2']

In [7]:
start = dt.datetime(2018, 2, 28)
end = dt.datetime(2018, 7, 1)

folders = []
for i in range(0,(end - start).days + 1,5):
    folders.append((start + dt.timedelta(days=i)).strftime('%d%b%y').lower())
for run in runs:
    path = f'/home/jvalenti/scratch/run_SHEM/{run}/'
    PPtot= []
    PPPB_ratio = []
    for folder in folders:
        fn = make_filename(path,folder, var='prod_T', res='d')
        with xr.open_dataset(fn) as ds:
            PPdiat = ds['PPDIAT'].isel(time_counter=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))*mask['e3t_0'].isel(t=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))
            PPphy = ds['PPPHY'].isel(time_counter=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))*mask['e3t_0'].isel(t=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))
            PPhbac = ds['HetHBAC'].isel(time_counter=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))*mask['e3t_0'].isel(t=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))
        PPPB_ratio.append(PPhbac.sum()/(PPdiat.sum()+PPphy.sum()))
    print(run,np.mean(PPPB_ratio))

SHEM18 1.620537962397085
tuning/diat_pref 0.6758172691637929
tuning/exc_hbac 1.5804533888610073
tuning/exc_hbac_2 0.4648169336618027
tuning/growth_flag 1.732894594800991
tuning/growth_flag_2 1.3647549112378785
tuning/mort_hbac 1.6598509611841437
tuning/pred_flag 1.4693552698911085
tuning/remin 0.3790947634741867
tuning/remin2 0.37390405191258863
tuning/predmine 0.3843901903783339
tuning/mort_hbac_2 1.5779941043252268
